In [0]:
import mlflow
import logging

logging.getLogger('mlflow').setLevel(logging.ERROR)
mlflow.pyspark.ml.autolog()

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb = ctx.notebookPath().get().split("/")[2]
print(nb)
safe_path = f"dbfs:/FileStore/{nb}/"
print(safe_path)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

experiment_name = f"/Shared/{nb}_occupancy_detection2"
print(experiment_name)
print()

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)
mlflow.set_experiment(experiment_name)

odl_user_2032998@databrickslabs.com
dbfs:/FileStore/odl_user_2032998@databrickslabs.com/
/Shared/odl_user_2032998@databrickslabs.com_occupancy_detection2



<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/3856807277166833', creation_time=1772630849708, experiment_id='3856807277166833', last_update_time=1772632403929, lifecycle_stage='active', name='/Shared/odl_user_2032998@databrickslabs.com_occupancy_detection2', tags={'mlflow.experiment.sourceName': '/Shared/odl_user_2032998@databrickslabs.com_occupancy_detection2',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'Group-41',
 'mlflow.ownerId': '82557566109249'}>

In [0]:
mainDF = spark.read.csv("/Volumes/teaching/datasets/raw_data/week_7/Occupancy_Detection_Data.csv", header=True, inferSchema=True)

In [0]:
occupancyDF = mainDF
display(occupancyDF)

Timestamp,Temperature,Humidity,CO2,HumidityRatio,Occupancy
11/02/2015 14:51,21.7675,31.1225,1009.5,0.005021569,1
11/02/2015 14:57,21.79,31.46333333,1027.333333,0.005084053,1
11/02/2015 15:04,21.89,31.6,1060.5,0.005137857,1
11/02/2015 15:19,21.89,31.73,1100.0,0.005159169,1
11/02/2015 15:36,21.89,30.65,896.5,0.004982159,1
11/02/2015 15:59,21.89,30.42666667,792.6666667,0.004945567,1
11/02/2015 16:01,21.89,30.39,797.75,0.00493956,1
11/02/2015 16:02,21.89,30.39,798.0,0.00493956,1
11/02/2015 16:05,21.89,30.35666667,797.5,0.004934099,1
11/02/2015 16:14,21.89,29.9175,756.0,0.00486216,1


Databricks visualization. Run in Databricks to view.

In [0]:
occupancyDF.createOrReplaceTempView("occupancyView")

In [0]:
%sql

SELECT Occupancy,
       AVG(Temperature) AS Average_Temperature,
       AVG(Humidity) AS Average_Humidity,
       AVG(CO2) AS Average_CO2,
       AVG(HumidityRatio) AS Average_Humidity_Ratio
FROM occupancyView
GROUP BY Occupancy

Occupancy,Average_Temperature,Average_Humidity,Average_CO2,Average_Humidity_Ratio
1,21.871518931764747,27.96006930359028,958.1930020277889,0.004539106375253549
0,20.58318598559928,28.040931237681857,622.8635232447936,0.0041981035245579525


In [0]:
%sql
SELECT Occupancy,
       COUNT(*) AS Row_Count,
       MAX(Temperature) AS Max_Temp,
       MIN(Temperature) AS Min_Temp,
       MAX(Humidity) AS Max_Humidity,
       MIN(Humidity) AS Min_Humidity,
       MAX(CO2) AS Max_CO2,
       MIN(CO2) AS Min_CO2,
       MAX(HumidityRatio) AS Max_Humidity_Ratio,
       MIN(HumidityRatio) AS Min_Humidity_Ratio
FROM occupancyView
GROUP BY Occupancy

Occupancy,Row_Count,Max_Temp,Min_Temp,Max_Humidity,Min_Humidity,Max_CO2,Min_CO2,Max_Humidity_Ratio,Min_Humidity_Ratio
1,493,24.2,19.575,39.1175,18.99333333,2023.0,472.0,0.006464333,0.002830085
0,509,24.34,19.1,39.0,17.0,1774.0,420.5,0.006176767,0.002674127


In [0]:
occupancyDF = occupancyDF.drop(occupancyDF.Timestamp)
occupancyDF.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- CO2: double (nullable = true)
 |-- HumidityRatio: double (nullable = true)
 |-- Occupancy: integer (nullable = true)



In [0]:
from pyspark.ml.feature import RFormula

preprocess = RFormula(formula="Occupancy ~ .")
occupancyDF = preprocess.fit(occupancyDF).transform(occupancyDF)

occupancyDF.show(15)

+-----------+-----------+-----------+-------------+---------+--------------------+-----+
|Temperature|   Humidity|        CO2|HumidityRatio|Occupancy|            features|label|
+-----------+-----------+-----------+-------------+---------+--------------------+-----+
|    21.7675|    31.1225|     1009.5|  0.005021569|        1|[21.7675,31.1225,...|  1.0|
|      21.79|31.46333333|1027.333333|  0.005084053|        1|[21.79,31.4633333...|  1.0|
|      21.89|       31.6|     1060.5|  0.005137857|        1|[21.89,31.6,1060....|  1.0|
|      21.89|      31.73|     1100.0|  0.005159169|        1|[21.89,31.73,1100...|  1.0|
|      21.89|      30.65|      896.5|  0.004982159|        1|[21.89,30.65,896....|  1.0|
|      21.89|30.42666667|792.6666667|  0.004945567|        1|[21.89,30.4266666...|  1.0|
|      21.89|      30.39|     797.75|   0.00493956|        1|[21.89,30.39,797....|  1.0|
|      21.89|      30.39|      798.0|   0.00493956|        1|[21.89,30.39,798....|  1.0|
|      21.89|30.35666

In [0]:
display(occupancyDF)

Temperature,Humidity,CO2,HumidityRatio,Occupancy,features,label
21.7675,31.1225,1009.5,0.005021569,1,"Map(vectorType -> dense, length -> 4, values -> List(21.7675, 31.1225, 1009.5, 0.005021569))",1.0
21.79,31.46333333,1027.333333,0.005084053,1,"Map(vectorType -> dense, length -> 4, values -> List(21.79, 31.46333333, 1027.333333, 0.005084053))",1.0
21.89,31.6,1060.5,0.005137857,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 31.6, 1060.5, 0.005137857))",1.0
21.89,31.73,1100.0,0.005159169,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 31.73, 1100.0, 0.005159169))",1.0
21.89,30.65,896.5,0.004982159,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 30.65, 896.5, 0.004982159))",1.0
21.89,30.42666667,792.6666667,0.004945567,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 30.42666667, 792.6666667, 0.004945567))",1.0
21.89,30.39,797.75,0.00493956,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 30.39, 797.75, 0.00493956))",1.0
21.89,30.39,798.0,0.00493956,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 30.39, 798.0, 0.00493956))",1.0
21.89,30.35666667,797.5,0.004934099,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 30.35666667, 797.5, 0.004934099))",1.0
21.89,29.9175,756.0,0.00486216,1,"Map(vectorType -> dense, length -> 4, values -> List(21.89, 29.9175, 756.0, 0.00486216))",1.0


In [0]:
(trainingDF, testDF) = occupancyDF.randomSplit([0.7, 0.3], seed=100)

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
model = dt.fit(trainingDF)

Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
predictions = model.transform(testDF)
predictions.show()

+-----------+-----------+-----------+-------------+---------+--------------------+-----+-------------+--------------------+----------+
|Temperature|   Humidity|        CO2|HumidityRatio|Occupancy|            features|label|rawPrediction|         probability|prediction|
+-----------+-----------+-----------+-------------+---------+--------------------+-----+-------------+--------------------+----------+
|       19.2|       30.7|      429.0|  0.004222155|        0|[19.2,30.7,429.0,...|  0.0|  [123.0,0.0]|           [1.0,0.0]|       0.0|
|     19.245|      31.65|      435.0|  0.004366035|        0|[19.245,31.65,435...|  0.0|  [123.0,0.0]|           [1.0,0.0]|       0.0|
|      19.26|31.46333333|431.6666667|  0.004344191|        0|[19.26,31.4633333...|  0.0|  [123.0,0.0]|           [1.0,0.0]|       0.0|
|      19.29|      26.79|      469.0|  0.003702057|        0|[19.29,26.79,469....|  0.0|  [123.0,0.0]|           [1.0,0.0]|       0.0|
|      19.29|       27.5|      432.0|   0.00380077|    

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Accuracy = {accuracy*100}%")

Accuracy = 89.22558922558923%


## Using Grid Search

In [0]:
from pyspark.ml.tuning import ParamGridBuilder

parameters = (
    ParamGridBuilder()
    .addGrid(dt.impurity, ["gini", "entropy"])
    .addGrid(dt.maxDepth, [3, 5, 7])
    .addGrid(dt.maxBins, [16, 32, 64])
    .build()
)


In [0]:
from pyspark.ml.tuning import TrainValidationSplit

tvs = (
    TrainValidationSplit()
    .setSeed(100)
    .setTrainRatio(0.75)
    .setEstimatorParamMaps(parameters)
    .setEstimator(dt)
    .setEvaluator(evaluator)
)

In [0]:
gridSearchModel = tvs.fit(trainingDF)

Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
bestModel = gridSearchModel.bestModel

print("Parameters for the best model")
print(f"MaxDepth: {bestModel.getMaxDepth()}")
print(f"MaxBins: {bestModel.getMaxBins()}")
print(f"Impurity: {bestModel.getImpurity()}")

Parameters for the best model
MaxDepth: 7
MaxBins: 32
Impurity: entropy


In [0]:
evaluator.evaluate(bestModel.transform(testDF))

0.9023569023569024

In [0]:
from pyspark.ml.feature import VectorAssembler
import mlflow.spark

model_uri = ""
model = mlflow.spark.load_model(model_uri)

input_sdf = spark.createDataFrame([{
    "Temperature" : 22.5,
    "Humidity": 31.0,
    "CO2": 800.0,
    "HumidityRatio": 0.0045,
}])

assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "CO2", "HumidityRatio"],
    outputCol="features",
)

input_with_features = assembler.transform(input_sdf)

predictions = model.transform(input_with_features)
predictions.select("prediction", "probability").show(truncate=False)